### Knowledge Graph


#### Standard RAG uses Vector Search, which works like searching a library by matching keywords or general meaning. It’s good for finding specific facts, but not for making connections. That’s where GraphRAG comes in. Instead of seeing your data as separate documents, GraphRAG views it as a network of connected facts.

#### GraphRAG uses a Knowledge Graph, which is a network of entities (nodes) and relationships (edges). This lets it move from one fact to another and find hidden connections that vector search can’t catch.

Pipeline:

1. Read text.
2. Extract relationships (Subject -> Predicate -> Object).
3. Build a Graph using NetworkX.
4. Retrieve context by walking the graph (Multi-hop reasoning).
5. Answer a question based on that deep context.

In [ ]:
# All needed imports

# Data Science Libraries
from huggingface_hub import InferenceClient

# Standard Libraries
import os
import networkx as nx
import json

/root/opt/la-i-b/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Must test models, cause we need a free tier model
client = InferenceClient(token=os.getenv("HF_TOKEN"))

models_to_test = [
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-2-9b-it",
    "HuggingFaceH4/zephyr-7b-beta",
]

for m in models_to_test:
    try:
        result = client.chat_completion(
            messages=[{"role": "user", "content": "Say hello"}],
            model=m,
            max_tokens=10
        )
        print(f"{m}: WORKS - {result.choices[0].message.content}")
    except Exception as e:
        print(f"{m}: FAILED - {str(e)[:80]}")

meta-llama/Meta-Llama-3-8B-Instruct: FAILED - (Request ID: Root=1-6a60a9b6-5b3ec5623dcd6bfd234f993b;c83c10be-d0ec-4a01-9ab5-e2
meta-llama/Meta-Llama-3.1-8B-Instruct: FAILED - (Request ID: Root=1-6a60a9b6-09844c4f41536ec519bae46c;2e7730c0-82f3-41b5-8e56-1c
mistralai/Mistral-7B-Instruct-v0.3: FAILED - (Request ID: Root=1-6a60a9b6-67bb573e2f44def35ef09e68;e253dcc2-c7f2-423a-90b8-61
mistralai/Mixtral-8x7B-Instruct-v0.1: FAILED - (Request ID: Root=1-6a60a9b7-517c609d0c2b0254650ff6fe;d087a7b1-22c2-4572-b134-13
Qwen/Qwen2.5-7B-Instruct: WORKS - Hello! How can I assist you today?
microsoft/Phi-3-mini-4k-instruct: FAILED - (Request ID: Root=1-6a60a9c1-2d89c1ee7db5960b00bf78e7;29162a73-190a-4279-8fe2-85
google/gemma-2-9b-it: FAILED - (Request ID: Root=1-6a60a9c1-691b66d176fd81867c5d677a;e019c522-457a-4a0c-8a54-4b
HuggingFaceH4/zephyr-7b-beta: FAILED - (Request ID: Root=1-6a60a9c1-72ac924900761c470e3d611b;0e5f2cad-cc90-49fe-a9e4-34


In [4]:
MODEL = "Qwen/Qwen2.5-7B-Instruct" # 

def ask_llm(prompt, max_tokens=500):
    """Helper to call the LLM"""
    result = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=max_tokens,
        temperature=0.0, # facts, no creativity
    )
    return result.choices[0].message.content.strip()

In [5]:
# Turnin' text into data : extracting triples from text
# We can’t put raw text straight into a graph; we need triples. 
# A triple is the basic unit of a knowledge graph: (Head) -> [Relation] -> (Tail).

def extract_triples(text):
    """Replaces the LangChain extraction_chain"""
    prompt = f"""You are an expert knowledge graph builder.
                Extract entities and relationships from the text.
                Return ONLY a JSON list. No explanation, no markdown, no backticks.
                Each item must contain:
                - "head": source entity
                - "relation": relationship
                - "tail": target entity

                Text:
                {text}

                Output JSON:"""
    
    response = ask_llm(prompt)
    # Clean up in case the model wraps in backticks
    response = response.strip().strip('`')
    if response.startswith('json'):
        response = response[4:]
    return json.loads(response)


text = """
The Moon orbits Earth.The Moon has an atmosphere called the Exosphere.Apollo 11 landed on the Moon.The Moon has a crater named the South Pole-Aitken Basin.Earth's Moon is classified as a natural satellite.
"""

print("\n Extracting knowledge graph triples...\n")
triples = extract_triples(text)
print(triples)


 Extracting knowledge graph triples...

[{'head': 'The Moon', 'relation': 'orbits', 'tail': 'Earth'}, {'head': 'The Moon', 'relation': 'has', 'tail': 'an atmosphere called the Exosphere'}, {'head': 'The Moon', 'relation': 'landed on', 'tail': 'Apollo 11'}, {'head': 'The Moon', 'relation': 'has', 'tail': 'a crater named the South Pole-Aitken Basin'}, {'head': "Earth's Moon", 'relation': 'classified as', 'tail': 'a natural satellite'}]


In [6]:
# Building Knowledge Graph
kg = nx.DiGraph() # DiGraph means "Directed Graph" (arrows point one way)

def build_knowledge_graph(triples):
    for item in triples:
        head = item.get("head")
        tail = item.get("tail")
        relation = item.get("relation")

        if head and tail:
            kg.add_node(head)
            kg.add_node(tail)
            kg.add_edge(head, tail, label=relation)

build_knowledge_graph(triples)

print("\n Nodes in Graph:")
print(list(kg.nodes()))


 Nodes in Graph:
['The Moon', 'Earth', 'an atmosphere called the Exosphere', 'Apollo 11', 'a crater named the South Pole-Aitken Basin', "Earth's Moon", 'a natural satellite']


In [7]:
# Multi-Hop Retrieval: Finding paths between entities
def retrieve_graph_context(entity, max_depth=2):
    context = set()
    visited_nodes = set()

    def dfs(node, depth):
        if depth > max_depth:
            return
        visited_nodes.add(node)

        # Checking Outgoing edges (What does this node do?)
        for neighbor in kg.successors(node):
            relation = kg.get_edge_data(node, neighbor)["label"]
            context.add(f"{node} {relation} {neighbor}")
            if neighbor not in visited_nodes:
                dfs(neighbor, depth + 1)

        # Checking Incoming edges (Who interacts with this node?)
        for predecessor in kg.predecessors(node):
            relation = kg.get_edge_data(predecessor, node)["label"]
            context.add(f"{predecessor} {relation} {node}")
            if predecessor not in visited_nodes:
                dfs(predecessor, depth + 1)

    if entity in kg.nodes:
        dfs(entity, 1) # Starts the traversal

    return ". ".join(context)

In [8]:
# RAG - Feeding that rich, interconnected context back to the LLM to answer the user’s question

def graph_rag_answer(entity, question, max_depth=3):
    """Replaces the LangChain rag_chain. Ask for a depth of at least 3 to catch distant connections"""
    graph_context = retrieve_graph_context(entity, max_depth=max_depth)
    
    print(f"\nRetrieved Graph Context:\n{graph_context}\n")
    
    prompt = f"""Answer the question using ONLY the context below.

                Context:
                {graph_context}

                Question:
                {question}

                Answer:"""
    
    return ask_llm(prompt, max_tokens=200)

In [10]:
# Asking a Multi-Hop reasoning question
def extract_entity_from_question(question, available_entities):
    prompt = f"""Given this question and this list of known entities, return ONLY the single entity name from the list that the question is most about. Return just the entity name, nothing else.

Entities: {available_entities}

Question: {question}

Entity:"""
    return ask_llm(prompt, max_tokens=30).strip()

question = "On which natural satellite did Apollo land?"

entity = extract_entity_from_question(question, list(kg.nodes()))

answer = graph_rag_answer(entity, question)

print(f"\nFinal Answer:\n{answer}")


Retrieved Graph Context:
Earth's Moon classified as a natural satellite


Final Answer:
Earth's Moon
